In [2]:
import pandas as pd
import requests
import pyodbc
from datetime import datetime


csv_file_path = 'egypt_traffic_points.csv' 

try:
 
    df = pd.read_csv(csv_file_path)
    
    
    column_name = 'City' 
    
  
    unique_cities = df[column_name].dropna().unique()
    
   
    cities_list = [f"{city},EG" for city in unique_cities]
    
    print(f"📌 تم العثور على {len(cities_list)} محافظات فريدة في الملف.")
    print(f"المحافظات هي: {unique_cities}\n")

except Exception as e:
    print(f"⚠️ حصل خطأ في قراءة ملف الـ CSV، اتأكد من اسم الملف واسم العمود: {e}")
    cities_list = []


if len(unique_cities) > 0: 
    API_KEY = "7bec552fd2f4e89cfce26bda17cc94e9"
    

    city_mapping = {
        "Gharbia": "Tanta",      
        "Qalyubia": "Banha"    
    }
    
    conn_str = (
        r'DRIVER={ODBC Driver 17 for SQL Server};'
        r'SERVER=DESKTOP-2A0QBS7\SQLEXPRESS;' 
        r'DATABASE=EgyptTrafficDB;'
        r'Trusted_Connection=yes;'
    )

    try:
        print("⏳ جاري الاتصال بقاعدة البيانات EgyptTrafficDB...")
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        print("✅ تم الاتصال بالسيرفر بنجاح!\n")
        
        for city in unique_cities:
      
            api_city = city_mapping.get(city, city)
            
            print(f"🔄 جاري سحب بيانات الطقس لـ {city} (باسم {api_city} للموقع)...")
            URL = f"http://api.openweathermap.org/data/2.5/weather?q={api_city},EG&appid={API_KEY}&units=metric"
            
            response = requests.get(URL)
            
            if response.status_code == 200:
                data = response.json()
                
                temp = data['main']['temp']
                humidity = data['main']['humidity']
                wind_speed = data['wind']['speed']
                weather_desc = data['weather'][0]['description']
                
          
                current_time = datetime.now()
                weather_date_hour = current_time.replace(minute=0, second=0, microsecond=0)
                
                print(f"   ☁️ الحالة: {temp}°C, {weather_desc}")
                
                insert_query = """
                    INSERT INTO WeatherData (City_Name, Weather_DateHour, Temperature_C, Humidity_Pct, Wind_Speed_ms, Weather_Condition)
                    VALUES (?, ?, ?, ?, ?, ?)
                """
              
                cursor.execute(insert_query, city, weather_date_hour, temp, humidity, wind_speed, weather_desc)
                conn.commit()
                print(f"    تم حفظ بيانات {city} بنجاح.\n")
                
            else:
                print(f" فشل سحب بيانات {city}: {response.text}\n")

        cursor.close()
        conn.close()
        print(" تم الانتهاء من تحديث الطقس لكل المحافظات اللي في الملف!")

    except Exception as e:
        print(f" حصل خطأ في الداتا بيز أو الـ API: {e}")

📌 تم العثور على 6 محافظات فريدة في الملف.
المحافظات هي: <StringArray>
['Alexandria', 'Beheira', 'Gharbia', 'Qalyubia', 'Cairo', 'Giza']
Length: 6, dtype: str

⏳ جاري الاتصال بقاعدة البيانات EgyptTrafficDB...
✅ تم الاتصال بالسيرفر بنجاح!

🔄 جاري سحب بيانات الطقس لـ Alexandria (باسم Alexandria للموقع)...
   ☁️ الحالة: 27.89°C, clear sky
    تم حفظ بيانات Alexandria بنجاح.

🔄 جاري سحب بيانات الطقس لـ Beheira (باسم Beheira للموقع)...
   ☁️ الحالة: 32.51°C, clear sky
    تم حفظ بيانات Beheira بنجاح.

🔄 جاري سحب بيانات الطقس لـ Gharbia (باسم Tanta للموقع)...
   ☁️ الحالة: 33.44°C, clear sky
    تم حفظ بيانات Gharbia بنجاح.

🔄 جاري سحب بيانات الطقس لـ Qalyubia (باسم Banha للموقع)...
   ☁️ الحالة: 34.21°C, clear sky
    تم حفظ بيانات Qalyubia بنجاح.

🔄 جاري سحب بيانات الطقس لـ Cairo (باسم Cairo للموقع)...
   ☁️ الحالة: 33.42°C, clear sky
    تم حفظ بيانات Cairo بنجاح.

🔄 جاري سحب بيانات الطقس لـ Giza (باسم Giza للموقع)...
   ☁️ الحالة: 33.41°C, clear sky
    تم حفظ بيانات Giza بنجاح.

 تم الان

In [ ]:
import time
from datetime import datetime
import requests
import pyodbc
import schedule
import pandas as pd


SERVER_NAME = r'DESKTOP-2A0QBS7\SQLEXPRESS'
DATABASE_NAME = 'EgyptTrafficDB'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;'

TOMTOM_API_KEY = "5cUh0bqAswvmUlzjXzwg2hAS7nDoG87L"
WEATHER_API_KEY = "7bec552fd2f4e89cfce26bda17cc94e9"

def run_all_data_collection():
    print(f"\n--- بدء دورة سحب الطقس والمرور الجديدة في: {datetime.now()} ---")
    
   
    csv_file_path = 'egypt_traffic_points.csv' 
    unique_cities = []
    
    try:
        df = pd.read_csv(csv_file_path)
        column_name = 'City' 
        unique_cities = df[column_name].dropna().unique()
        cities_list = [f"{city},EG" for city in unique_cities]
        print(f" تم العثور على {len(cities_list)} محافظات فريدة في الملف.")
        print(f"المحافظات هي: {unique_cities}\n")
    except Exception as e:
        print(f" حصل خطأ في قراءة ملف الـ CSV: {e}")

    if len(unique_cities) > 0:
        city_mapping = {
            "Gharbia": "Tanta",
            "Qalyubia": "Banha"
        }
        
        try:
            print(" جاري الاتصال بقاعدة البيانات لادخال الطقس...")
            conn = pyodbc.connect(conn_str)
            cursor = conn.cursor()
            print(" تم الاتصال بالسيرفر بنجاح!\n")
            
            for city in unique_cities:
                api_city = city_mapping.get(city, city)
                print(f" جاري سحب بيانات الطقس لـ {city} (باسم {api_city} للموقع)...")
                URL = f"http://api.openweathermap.org/data/2.5/weather?q={api_city},EG&appid={WEATHER_API_KEY}&units=metric"
                
                response = requests.get(URL)
                
                if response.status_code == 200:
                    data = response.json()
                    temp = data['main']['temp']
                    humidity = data['main']['humidity']
                    wind_speed = data['wind']['speed']
                    weather_desc = data['weather'][0]['description']
                    
                    current_time = datetime.now()
                    weather_date_hour = current_time.replace(minute=0, second=0, microsecond=0)
                    
                    print(f"   ☁️ الحالة: {temp}°C, {weather_desc}")
                    
                    insert_query = """
                        INSERT INTO WeatherData (City_Name, Weather_DateHour, Temperature_C, Humidity_Pct, Wind_Speed_ms, Weather_Condition)
                        VALUES (?, ?, ?, ?, ?, ?)
                    """
                    cursor.execute(insert_query, city, weather_date_hour, temp, humidity, wind_speed, weather_desc)
                    conn.commit()
                    print(f"    تم حفظ بيانات {city} بنجاح.\n")
                else:
                    print(f" فشل سحب بيانات {city}: {response.text}\n")
                    
            cursor.close()
            conn.close()
            print(" تم الانتهاء من تحديث الطقس لكل المحافظات اللي في الملف!")
        except Exception as e:
            print(f" حصل خطأ في الداتا بيز أو الـ API الخاص بالطقس: {e}")

    
    try:
        print("\n بدء دورة سحب المرور...")
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        
        cursor.execute("SELECT LocationID, Latitude, Longitude, LocationName FROM Locations")
        locations = cursor.fetchall()
        
        success_count = 0
        for loc in locations:
            loc_id, lat, lon, name = loc
            url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat}%2C{lon}&key={TOMTOM_API_KEY}"
            
            try:
                response = requests.get(url)
                if response.status_code == 200:
                    data = response.json()
                    flow_data = data['flowSegmentData']
                    current_speed = flow_data['currentSpeed']
                    normal_speed = flow_data['freeFlowSpeed']
                    
                    if current_speed < (normal_speed * 0.6):
                        status = "Heavy Congestion"
                    elif current_speed < normal_speed:
                        status = "Moderate"
                    else:
                        status = "Free Flow"
                    
                    insert_query = """
                        INSERT INTO Traffic_Logs (LocationID, CurrentSpeed, NormalSpeed, TrafficStatus, RecordedAt)
                        VALUES (?, ?, ?, ?, ?)
                    """
                    cursor.execute(insert_query, (loc_id, current_speed, normal_speed, status, datetime.now()))
                    conn.commit()
                    success_count += 1
                else:
                    print(f"فشل سحب نقطة {name}: {response.status_code}")
            except Exception as e:
                print(f"خطأ في نقطة {name}: {e}")
                
            time.sleep(0.5)
            
        conn.close()
        print(f"خلصت دورة السحب بنجاح. تم تسجيل {success_count} نقطة في الـ DB.")
    except Exception as e:
        print(f" حصل خطأ في الاتصال بقاعدة البيانات للمرور: {e}")


# 1. 05:00 ص
schedule.every().day.at("05:00").do(run_all_data_collection)
# 2. 08:00 ص
schedule.every().day.at("08:00").do(run_all_data_collection)
# 3. 12:00 م
schedule.every().day.at("12:00").do(run_all_data_collection)
# 4. 05:00 م
schedule.every().day.at("17:00").do(run_all_data_collection)
# 5. 10:00 م
schedule.every().day.at("22:00").do(run_all_data_collection)
# 6. 02:00 ص
schedule.every().day.at("02:00").do(run_all_data_collection)

print("الـ Pipeline مجمعة وشغالة في الخلفية ومستنية المواعيد المحددة!")

while True:
    schedule.run_pending()
    time.sleep(60)